# Figure 9.15: Geometric Intuition for L1, L2, and Elastic Net

Constraint shape changes where the loss contours are touched and how the path approaches the optimum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def ellipse(X, Y):
    return 0.7 * (X - 0.8) ** 2 + 1.25 * (Y - 0.2) ** 2 + 0.35 * (X - 0.8) * (Y - 0.2)


def l1_shape(scale):
    return np.array([0, scale, 0, -scale, 0]), np.array([scale, 0, -scale, 0, scale])


def l2_shape(scale):
    t = np.linspace(0, 2 * np.pi, 200)
    return scale * np.cos(t), scale * np.sin(t)


def elastic_shape(scale, rho):
    t = np.linspace(0, 2 * np.pi, 200)
    p = 1.15 + 1.5 * (1 - rho)
    x = scale * np.sign(np.cos(t)) * np.abs(np.cos(t)) ** (2 / p)
    y = scale * np.sign(np.sin(t)) * np.abs(np.sin(t)) ** (2 / p)
    return x, y


def path_points(target, steps, step_scale):
    pts = [np.array([2.0, 1.8])]
    for _ in range(1, steps):
        pts.append(pts[-1] + step_scale * (np.array(target) - pts[-1]))
    pts = np.stack(pts, axis=0)
    return pts[:, 0], pts[:, 1]


def draw(scale=1.25, rho=0.35, steps=12, path_scale=0.60):
    xs = np.linspace(-2.6, 2.6, 220)
    ys = np.linspace(-2.4, 2.4, 220)
    X, Y = np.meshgrid(xs, ys)
    Z = ellipse(X, Y)
    fig, axs = plt.subplots(2, 2, figsize=(10.0, 8.0), constrained_layout=True)
    panels = [
        ("L1 geometry", l1_shape(scale), (0.0, scale * 0.72), "#dc2626"),
        ("Lasso path", l1_shape(scale), (0.0, scale * 0.72), "#dc2626"),
        ("L2 geometry", l2_shape(scale), (scale * 0.72, scale * 0.38), "#2563eb"),
        ("Elastic Net path", elastic_shape(scale, rho), (scale * 0.55, scale * 0.36), "#7c3aed"),
    ]
    for ax, (title, shape, optimum, color) in zip(axs.ravel(), panels):
        ax.contour(X, Y, Z, levels=12, cmap="viridis")
        ax.plot(shape[0], shape[1], color="black", lw=2)
        px, py = path_points(optimum, steps, path_scale)
        ax.plot(px, py, "-o", color=color, lw=2, ms=4)
        ax.scatter(*optimum, color=color, s=70, marker="D")
        ax.set_title(title)
        ax.set_xlim(-2.6, 2.6)
        ax.set_ylim(-2.4, 2.4)
        ax.axhline(0, color="0.6", lw=0.6)
        ax.axvline(0, color="0.6", lw=0.6)
        ax.set_aspect("equal", "box")
    plt.show()

widgets.interact(
    draw,
    scale=widgets.FloatSlider(min=0.6, max=2.2, step=0.05, value=1.25, continuous_update=False),
    rho=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.35, continuous_update=False),
    steps=widgets.IntSlider(min=4, max=24, step=1, value=12, continuous_update=False),
    path_scale=widgets.FloatSlider(min=0.2, max=1.2, step=0.05, value=0.60, continuous_update=False),
)
